# Etape1

In [1]:
import pandas as pd
df_customers = pd.read_csv("../../dataset/customers.csv")
df_transactions = pd.read_csv("../../dataset/transactions.csv")

- Combien de clients ? 
- Combien de transactions ? Sur quelle période ?
Quelles colonnes contiennent des valeurs manquantes, et dans quelle proportion ?
Quels types de données sont attendus vs observés ?
Y a-t-il des doublons ?

In [ ]:
m = df_customers["customer_id"].nunique()
e = df_transactions["invoice_id"].nunique()
min = df_transactions['invoice_date'].min()
max = df_transactions['invoice_date'].max() 


print(f'Nombre clients uniques : {m}')   
print(f'Nombre transactions uniques : {e}')
print(f'Min periode : {min}')      
print(f'Max periode : {max}')

Nombre clients uniques : 50000
Nombre transactions uniques : 253545
Min periode : 2007-07-06 12:20:00
Max periode : 2011-12-09 12:50:00


Quelles colonnes contiennent des valeurs manquantes, et dans quelle proportion ?


In [13]:
missing_transactions = pd.DataFrame({
    "missing_count": df_transactions.isna().sum(),
    "missing_percent": df_transactions.isna().mean() * 100
})

print(missing_transactions)

              missing_count  missing_percent
invoice_id                0         0.000000
customer_id          418258        22.766838
product_code              0         0.000000
product_name           7542         0.410530
quantity              16187         0.881099
unit_price                0         0.000000
invoice_date              0         0.000000
country                   0         0.000000


In [32]:
missing_customers = pd.DataFrame({
    "missing_count": df_customers.isna().sum(),
    "missing_percent": df_customers.isna().mean() * 100
})

print(missing_customers)

                missing_count  missing_percent
customer_id                 0              0.0
country                     0              0.0
first_purchase              0              0.0
last_purchase               0              0.0
n_orders                    0              0.0
total_spent                 0              0.0
avg_basket                  0              0.0
recency_days                0              0.0
tenure_days                 0              0.0


Quels types de données sont attendus vs observés ?


In [17]:
df_customers.dtypes

customer_id         int64
country            object
first_purchase     object
last_purchase      object
n_orders          float64
total_spent       float64
avg_basket        float64
recency_days      float64
tenure_days       float64
dtype: object

In [18]:
df_transactions.dtypes

invoice_id       object
customer_id     float64
product_code     object
product_name     object
quantity        float64
unit_price      float64
invoice_date     object
country          object
dtype: object

On trouve des strings et des floats/int. Il y une variation des types qui ne sont pas logiques : ex -> customer_id en integer pour les données client, customer_id en float sur les données de transaction. L'ID des commandes est en string. Pour ce qui est du reste, les nombres qui nous importent sont notamment les données des achats clients et les informations sur les articles. Qui sont tous en float. On doit convertir les nombres en int afin de pouvoir les analyser et détecter les erreurs, on retrouve du texte dans des colonnes où il ne devrait pas y en avoir et on a des nombres de commmande qui sont avec des décimales, ce qui est normalement impossible.

Y a-t-il des doublons ?


In [31]:
duplicated_customers = df_customers.duplicated().sum()
print(f'Nombre de doublons dans customers : {duplicated_customers}')

Nombre de doublons dans customers : 0


In [30]:
duplicated_transactions = df_transactions.duplicated().sum()
print(f'Nombre de doublons dans transactions : {duplicated_transactions}')

Nombre de doublons dans transactions : 34522


Data quality report :

transactions.csv : 
- Doublons lourdement identifiés (34 522)
- Variable ( quantity, nb_order, customer_id) en float 
- Customer vide (22%)

customers.csv : 
- RAS

# Etape 2

- Les customer_id manquants: que représentent ces lignes ? Faut-il les supprimer ou les conserver ?

- Les quantity négatives: d'où viennent-elles ? Quel impact sur les agrégations futures ?

- Les unit_price à zéro: anomalie ou cas métier légitime ?

- Les product_code atypiques: identifiez les codes non-produits (frais de port, ajustements comptables, etc.)

- Calculez line_total = quantity × unit_price. Vérifiez la cohérence.

In [50]:
df_transactions[
    (df_transactions["invoice_id"] == "581582") &
    (df_transactions["invoice_date"].str.startswith("2010-04-14"))
]

#Il faut conserver les customer_id manquants, ils correspondent des transactions existantes

,invoice_id,customer_id,product_code,product_name,quantity,unit_price,invoice_date,country
464139,581582,18288.0,21335,JUMBO BAG RED RETROSPOT,4.0,1.95,2010-04-14 12:20:00,United Kingdom
589869,581582,18288.0,90089,PINK ROSE WASHBAG,4.0,3.75,2010-04-14 12:20:00,United Kingdom
1291616,581582,18288.0,84791,SET OF 4 ENGLISH ROSE PLACEMATS,4.0,3.75,2010-04-14 12:20:00,United Kingdom
1558043,581582,NaN,22356,PINK HAPPY BIRTHDAY BUNTING,6.0,5.45,2010-04-14 12:20:00,United Kingdom
1638033,581582,18288.0,48188,PACK 20 ENGLISH ROSE PAPER NAPKINS,36.0,0.85,2010-04-14 12:20:00,United Kingdom
1821472,581582,18288.0,21983,JUMBO BAG BAROQUE BLACK WHITE,10.0,1.95,2010-04-14 12:20:00,United Kingdom


In [ ]:
df_transactions[
    (df_transactions["customer_id"] == "581582") &
    (df_transactions["invoice_date"].str.startswith("2010-04-14"))
]

#Il faut conserver les customer_id manquants, ils correspondent des transactions existantes

In [62]:
df_transactions[
    (df_transactions["customer_id"] == 14292)
    & (df_transactions["product_code"] == "22752")]

#Les quantités négatives viennent des clients qui rendent les produits qu'ils ont achetés
#Il ne faut donc pas prendre en compte ses valeurs dans un calcul, 
# les exclure pour analyser uniquement les ventes

,invoice_id,customer_id,product_code,product_name,quantity,unit_price,invoice_date,country
74868,C525552,14292.0,22752,SET 7 BABUSHKA NESTING BOXES,-1.0,8.50,2010-10-05 19:41:00,United Kingdom
263356,C530186,14292.0,22752,SET 7 BABUSHKA NESTING BOXES,-24.0,7.65,2010-11-02 10:57:00,United Kingdom
711942,523996,14292.0,22752,SET 7 BABUSHKA NESTING BOXES,24.0,7.65,2010-09-26 15:24:00,United Kingdom


In [68]:
zero_price = df_transactions[df_transactions["unit_price"] == 0]
print("Nombre de prix à zéro :", len(zero_price))


zero_price.head()

#La plupart des produits sans unit_price sont des fausses commandes
#hypothèse : commande annulée ?

Nombre de prix à zéro : 10674


,invoice_id,customer_id,product_code,product_name,quantity,unit_price,invoice_date,country
26,768176,NaN,22990,WHITE CHERRY LIGHTS,24.0,0.0,2009-06-08 12:20:00,United Kingdom
113,548380,NaN,22127,Dotcom sold in 6's,-60.0,0.0,2011-03-30 16:44:00,United Kingdom
136,734573,44742.0,21145,CHARLIE + LOLA RED HOT WATER BOTTLE,1.0,0.0,2010-12-29 12:20:00,United Kingdom
252,519368,NaN,37449,NaN,7.0,0.0,2010-08-16 13:09:00,United Kingdom
261,508700,NaN,22192,NaN,-1.0,0.0,2010-05-17 17:05:00,United Kingdom


In [77]:
print(df_transactions.loc[
    df_transactions["product_name"].notna() &
    (df_transactions["product_name"] == df_transactions["product_name"].str.lower()),
    "product_name"
].unique()[:10])

df_codes_mauvais = df_transactions.loc[
    df_transactions["product_name"].notna() &
    (df_transactions["product_name"] == df_transactions["product_name"].str.lower()),
    "product_name"
].unique()


print(f"Il y a {len(df_codes_mauvais)} product_code atypiques")

['adjustment' 'faulty' 'checked' 'damages'
 'to push order througha s stock was ' 'mailout' 'wrong ctn size' '?'
 'damaged' 'came coded as 20713']
Il y a 206 product_code atypiques


- Cohérence des dates ( first_purchase ≤ last_purchase )
- Valeurs aberrantes dans total_spent , n_orders , avg_basket
- Clients avec une seule transaction vs clients récurrents: quelle proportion ?

In [ ]:
customer_dates = df_transactions.groupby("customer_id").agg(
    first_purchase=("invoice_date", "min"),
    last_purchase=("invoice_date", "max")
)

incoherent_dates = customer_dates[
    customer_dates["first_purchase"] > customer_dates["last_purchase"]
]

print("Incohérences :", len(incoherent_dates))

Incohérences : 0


In [89]:
def remove_outliers_iqr(df, cols, factor=1.5):
    for col in cols:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        df = df[(df[col] >= Q1 - factor * IQR) & (df[col] <= Q3 + factor * IQR)]
    return df

cols = ["total_spent", "n_orders", "avg_basket"]

df_clean = remove_outliers_iqr(
    df_customers[(df_customers[cols] > 0).all(axis=1)].copy(),
    cols
)

In [90]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

cols = ["total_spent", "n_orders", "avg_basket"]
names = ["Dépenses totales", "Nombre de commandes", "Panier moyen"]
colors = ["#636EFA", "#EF553B", "#00CC96"]

fig = make_subplots(rows=1, cols=3, subplot_titles=names)

for i, (col, name, color) in enumerate(zip(cols, names, colors), start=1):
    fig.add_trace(
        go.Box(y=df_customers[col], name=name, marker_color=color, showlegend=False),
        row=1, col=i
    )

fig.update_layout(title="Distribution par groupe")
fig.show()

In [92]:
cols = ["total_spent", "n_orders", "avg_basket"]
names = ["Dépenses totales", "Nombre de commandes", "Panier moyen"]
colors = ["#636EFA", "#EF553B", "#00CC96"]

fig = make_subplots(rows=1, cols=3, subplot_titles=names)

for i, (col, name, color) in enumerate(zip(cols, names, colors), start=1):
    fig.add_trace(
        go.Box(y=df_clean[col], name=name, marker_color=color, showlegend=False),
        row=1, col=i
    )

fig.update_layout(title="Distribution par groupe")
fig.show()

# Etape 3